In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, lit
from pyspark.sql.types import TimestampType


In [0]:
dbutils.widgets.text("catalog_name", "")
catalog_name = dbutils.widgets.get("catalog_name")
dbutils.widgets.text("schema_bronze_name", "")
schema_bronze_name = dbutils.widgets.get("schema_bronze_name")
dbutils.widgets.text("schema_silver_name", "")
schema_silver_name = dbutils.widgets.get("schema_silver_name")

In [0]:
bronze_table = f"{catalog_name}.{schema_bronze_name}.crime_evh_bronze"
checkpoint_silver_loc = f"/Volumes/{catalog_name}/{schema_silver_name}/silver/checkpoints/"
target_silver_scd1 = f"{catalog_name}.{schema_silver_name}.crime_silver_scd1"
target_silver_scd2 = f"{catalog_name}.{schema_silver_name}.crime_silver_scd2"

In [0]:
bronze_stream_df = (
    spark.readStream.format("delta")     
    .table(bronze_table)    
)

In [0]:
silver_table = (bronze_stream_df.select("value.*", bronze_stream_df.timestamp.alias("bronze_ingestion_time"))\
    .withColumn("silver_ingestion_time", F.current_timestamp())    
)

#### empty strings into nulls

In [0]:
silver_cols = silver_table.columns

def blank_as_null(x):    
    return when(col(x) != "", col(x)).otherwise(None)

#### Not changing for timestamps because we have null values

In [0]:
updates = {f"{col_name}": blank_as_null(col_name) for col_name in silver_cols if not isinstance(silver_table.schema[col_name].dataType, TimestampType)}   
silver_table = silver_table.withColumns(updates)

In [0]:
columns_to_cast = {
    "suspect_age": "double",
    "latitude": "double",
    "longitude": "double",
    "victim_age": "double",
    "num_arrests": "double",
    "property_loss_usd": "double"
}

for col_name, target_type in columns_to_cast.items():
    if col_name in silver_cols:
        silver_table = silver_table.withColumn(col_name, F.col(col_name).try_cast(target_type))

### SCD1 (when data duplicated in different batches)
Supports schema evolution

In [0]:
def process_batch(microBatchDF, batchId):
  
  deduped_silver_df = microBatchDF.dropDuplicates(["incident_id"])   # deleting duplicates in one batch
  deduped_silver_df.createOrReplaceTempView("deduped_silver_df")

  spark.sql(f"""
      CREATE TABLE IF NOT EXISTS {target_silver_scd1}
      USING DELTA AS
      SELECT * FROM deduped_silver_df WHERE 1 = 0
  """)

  spark.sql(
  f"""
  MERGE WITH SCHEMA EVOLUTION INTO {target_silver_scd1} TARGET
  USING deduped_silver_df SOURCE
  ON TARGET.incident_id = SOURCE.incident_id
  WHEN MATCHED THEN
    UPDATE SET *
      
  WHEN NOT MATCHED THEN
    INSERT *    
  """
  )

In [0]:
query = (silver_table
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", f"{checkpoint_silver_loc}/scd1")
    .trigger(availableNow=True)
    .start()
)

### SCD2 (when data duplicated in different batches)

In [0]:
def process_batch_scd2(microBatchDF, batchId):
  
  deduped_silver_df = microBatchDF.dropDuplicates(["incident_id"])   # deleting duplicates in one batch
  deduped_silver_df.createOrReplaceTempView("deduped_silver_df")

  spark.sql(f"""
      CREATE TABLE IF NOT EXISTS {target_silver_scd2}
      USING DELTA AS
      SELECT * FROM deduped_silver_df WHERE 1 = 0
  """)

  spark.sql(
  f"""
    MERGE INTO {target_silver_scd2} AS target
    USING deduped_silver_df AS source
    ON target.incident_id = source.incident_id AND target.is_current = True

    WHEN MATCHED THEN UPDATE SET        
        is_current = False   
  """)


  spark.sql(
  f"""
    INSERT INTO {target_silver_scd2}
    (
      incident_id,
      crime_type,
      district,
      city,
      state,
      address,
      latitude,
      longitude, 
      incident_datetime,
      officer_id,
      officer_first_name,
      officer_last_name,
      badge_number,
      suspect_id,
      suspect_first_name,
      suspect_last_name,
      suspect_age,
      suspect_gender,
      suspect_race,
      victim_id,
      victim_first_name,
      victim_last_name,
      victim_age,
      victim_gender,
      victim_phone,
      weapon_used,
      severity,
      case_status,
      resolution,
      num_arrests,
      property_loss_usd,
      reported_online,
      notes,
      bronze_ingestion_time,
      silver_ingestion_time,
      is_current
    )    
    SELECT * FROM deduped_silver_df   
  """)


In [0]:
query = (silver_table.withColumn("is_current", lit(True))
    .writeStream
    .foreachBatch(process_batch_scd2)
    .option("checkpointLocation", f"{checkpoint_silver_loc}/scd2")    
    .trigger(availableNow=True)
    .start()
)

### Delta column mapping

In [0]:
# spark.sql(f"""
# ALTER TABLE {target_silver_scd1} SET TBLPROPERTIES (
#   'delta.minReaderVersion' = '2',
#   'delta.minWriterVersion' = '5',
#   'delta.columnMapping.mode' = 'name'
# )
# """)

In [0]:
# spark.sql(f"""
# ALTER TABLE {target_silver_scd2} SET TBLPROPERTIES (
#   'delta.minReaderVersion' = '2',
#   'delta.minWriterVersion' = '5',
#   'delta.columnMapping.mode' = 'name'
# )
# """)

In [0]:
# spark.sql(f"""
# ALTER TABLE {target_silver_scd1} RENAME COLUMN city TO city_name
# """)

# spark.sql(f"""
# ALTER TABLE {target_silver_scd2} RENAME COLUMN city TO city_name
# """)

### Table layout and optimization

In [0]:
# spark.sql(f"OPTIMIZE {target_silver_scd1}")
# spark.sql(f"OPTIMIZE {target_silver_scd2}")

In [0]:
# spark.sql(f"OPTIMIZE {target_silver_scd1} ZORDER BY (incident_id, incident_datetime)")
# spark.sql(f"OPTIMIZE {target_silver_scd2} ZORDER BY (incident_id, incident_datetime)")

In [0]:
# spark.sql(f"""
# ALTER TABLE {target_silver_scd1}
# CLUSTER BY (incident_id, incident_datetime)         
# """)

# spark.sql(f"""
# ALTER TABLE {target_silver_scd2}
# CLUSTER BY (incident_id, incident_datetime)         
# """)

In [0]:
# spark.sql(f"VACUUM {target_silver_scd1} RETAIN 168 HOURS") 
# spark.sql(f"VACUUM {target_silver_scd2} RETAIN 168 HOURS") 